# Phase 5: Cross-stage and cross-language error analysis

Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 5: "where do the models fail (ORG spans? Bengali script
tokens? LLM format drift?)".

This notebook trains nothing and calls no LLM. It reloads the saved models, scores the full gold test sets, and compares
**which entities each stage gets wrong**.

| Question | Systems | Test set |
| --- | --- | --- |
| Where does each stage fail on Hindi? | BiLSTM-CRF, MuRIL, IndicBERT, LLM zero-shot and few-shot | 864 Hindi gold sentences |
| What changes between zero-shot and in-language models? | Hindi-trained vs in-language MuRIL, seeds 42, 43, 44 | Marathi and Bengali gold |
| Do Bengali script tokens explain the errors? | Same MuRIL models | Hindi, Marathi, Bengali gold |

**Safety rules**

- **`results/results.csv` is read only.** The notebook stops if its hash changes.
- **Every system must reproduce its logged F1 within 0.001** before any error is analyzed.
- LLM predictions are read from the committed Phase 5 run. The LLM is not run again.
- Outputs go to `results/error_analysis/`.

**How to run (Kaggle)**

1. Commit and push this notebook to `phase5-cross-stage-errors`, because the run clones that branch.
2. Import this notebook with **GPU T4 x2** and **Internet** on, and attach the `GH_TOKEN` Secret. The Phase 2
   `best.pt` is downloaded from its public Drive folder and verified by SHA-256.
3. With `DEBUG = True`, use **Save & Run All**. This scores 40 sentences per language. Keep nothing from it.
4. Set `DEBUG = False` and use **Save & Run All** again.
5. Download `NaamRekha/results/error_analysis/` and the executed notebook, unchanged.

In [1]:
# Same pins as Phases 3 to 5, plus pytorch-crf from Phase 2.
%pip install -q transformers==5.22.1 accelerate==1.15.0 seqeval==1.2.2 sentencepiece==0.2.2 pytorch-crf==0.7.2  # Jupyter magic: quietly (-q) installs the exact pinned library versions this notebook was validated against, including pytorch-crf for the BiLSTM-CRF decoder reloaded later

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64  # used below to base64-encode a temporary git HTTP auth header (git's "basic" auth needs base64)
import os  # used for environment variables, path checks, and filesystem helpers
import subprocess  # used to shell out to `git clone` and `git rev-parse` for reproducibility tracking
import sys  # used to add the repo root to sys.path so `from src import ...` works

# Must be set before torch is imported.
## Why here, not later: CUDA_VISIBLE_DEVICES is read once by the CUDA driver the first time anything touches a GPU (e.g. torch's lazy init on first use); setting it after torch is imported would be a no-op and both GPUs would stay visible.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # restrict this process to GPU 0 only; must happen before torch initializes CUDA, hence it's set right after importing os

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"  # canonical repo URL used to clone the project on Kaggle/Colab
REPO_BRANCH = "phase5-cross-stage-errors"  # the branch holding this exact notebook, so the clone matches what's running here


def get_secret(name):  # looks up a named secret (e.g. GH_TOKEN) across the three environments this notebook can run in
    # Kaggle Secret, then Colab Secret, then environment variable.
    try:  # attempt the Kaggle-specific secret lookup first
        from kaggle_secrets import UserSecretsClient  # only importable inside a Kaggle kernel
        return UserSecretsClient().get_secret(name)  # fetch the secret from Kaggle's attached Secrets store
    except Exception:  # not on Kaggle, or secret not attached: fall through
        pass  # not on Kaggle (or secret not attached) - fall through to the next environment
    try:  # attempt the Colab-specific secret lookup next
        from google.colab import userdata  # only importable inside Google Colab
        return userdata.get(name)  # fetch the secret from Colab's userdata secrets store
    except Exception:  # not on Colab either: fall back to a plain environment variable
        return os.environ.get(name)  # local/plain environment: fall back to a plain env var, or None if unset


# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook. Check for a repo file, not
# just src/, because Kaggle has its own unrelated /kaggle/src folder.
## Why check for a specific file, not just `os.path.isdir("../src")`: Kaggle ships its own unrelated /kaggle/src directory, so testing for this repo's real src/data.py is what actually distinguishes "the project is already here" from "some other src folder exists".
if os.path.isfile("../src/data.py"):  # running locally from notebooks/: the real src/data.py sits one level up
    REPO_ROOT = os.path.abspath("..")  # repo root is the parent directory
elif os.path.isfile("src/data.py"):  # running with cwd already at the repo root (e.g. repo root opened directly)
    REPO_ROOT = os.path.abspath(".")  # repo root is the current directory
else:  # not running locally with the repo already present: must be Kaggle/Colab, so clone it
    REPO_ROOT = os.path.abspath("NaamRekha")  # Kaggle/Colab: the repo will be cloned into ./NaamRekha next to the notebook
    if not os.path.isdir(REPO_ROOT):  # only clone if it isn't already there (idempotent re-runs)
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # copy the environment and disable git's interactive credential prompt
        token = get_secret("GH_TOKEN")  # optional read-only GitHub token, needed only if the repo/branch requires auth
        if token:  # only build the auth header if a token secret was actually found
            # Passed through the environment so it is never saved in .git/config
            # (Kaggle keeps that folder as output) or printed if the clone fails.
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()  # build the base64 "user:token" blob HTTP basic auth expects
## Why inject the auth header via GIT_CONFIG_* environment variables instead of putting the token in REPO_URL or writing it to .git/config: Kaggle persists .git/config as part of the kernel's saved output, so a token written to disk there would leak into the published notebook output; an env var scoped to this one subprocess call is never written to disk or printed.
            env.update(GIT_CONFIG_COUNT="1",  # tells git to apply exactly one extra -c style config entry from the environment
                       GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",  # the config key: an extra HTTP header for github.com requests
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")  # the header value: HTTP Basic auth carrying the token
## Why --depth 1 --branch REPO_BRANCH instead of a full clone: this notebook only needs the tip of one branch to run, so a shallow single-branch clone is faster and lighter on Kaggle/Colab's limited disk and network - at the cost of not having full commit history available inside the run, which isn't needed here.
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],  # shallow single-branch clone to save time/bandwidth
                       check=True, env=env)  # raise if the clone fails; use the env with the auth header injected
sys.path.insert(0, REPO_ROOT)  # make the repo root importable so `from src import ...` resolves to the cloned/local copy

hf_token = get_secret("HF_TOKEN")      # optional: every model used here is public
if hf_token:  # only set HF_TOKEN if a Hugging Face token secret was actually found
    os.environ["HF_TOKEN"] = hf_token  # exposes the token to the `transformers`/`huggingface_hub` libraries via the standard env var

GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_ROOT, "rev-parse", "--short", "HEAD"], text=True).strip()  # records the exact commit this run used, for reproducibility logging

from src import LABELS, evaluate, id2label, label2id, load_split  # shared project code: label list, seqeval-based scorer, label<->id maps, and the CoNLL-style split loader

print("repo:", REPO_ROOT, "| commit:", GIT_COMMIT)  # sanity-check print: confirms which repo copy and commit are in use
print("labels:", LABELS)  # sanity-check print: confirms the label set (e.g. O, B-PER, I-PER, B-ORG, ...) matches expectations

Cloning into '/kaggle/working/NaamRekha'...


repo: /kaggle/working/NaamRekha | commit: a53f15e
labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


## Config

Model links and expected scores are read from `results/results.csv`, so they are not retyped here.

In [3]:
DEBUG = False              # True: 40 sentences per language, nothing kept. False: the real run.
DEBUG_SENTENCES = 40  # how many sentences per language DEBUG mode scores, for a fast smoke test

SOURCE_LANG = "hi"  # Hindi: the language every model in this project is originally trained on
TARGET_LANGS = ["mr", "bn"]  # Marathi and Bengali: the cross-lingual transfer targets being analyzed
LANGS = [SOURCE_LANG, *TARGET_LANGS]  # convenience list of all three languages evaluated in this notebook
## Why three seeds instead of one: a single seed can't tell a genuine zero-shot-vs-in-language effect apart from ordinary training-run noise (random init, data shuffling order); three seeds let `same_sign` later in the notebook check whether the effect holds in the same direction every time, which one run never could.
SEEDS = [42, 43, 44]  # the three training seeds used for the in-language/zero-shot MuRIL comparison, to separate real transfer effects from training-run noise
NATIVE_SCRIPT = {"hi": "Devanagari", "mr": "Devanagari", "bn": "Bengali"}  # each language's native writing system, used later to check if script mismatch explains errors

MAX_LENGTH = 128  # max subword sequence length passed to transformer tokenizers (sentences longer than this get truncated)
EVAL_BATCH_SIZE = 32  # batch size used when running transformer models for prediction
## Why a tolerance instead of exact equality: scoring under fp16 (as Phase 3 did) is not bit-exact across reruns/hardware, so comparing floats with `==` would spuriously fail the reproduction check even when nothing is actually wrong; 0.001 is chosen to be smaller than any real regression but larger than fp16 noise.
REPRO_TOLERANCE = 0.001   # about two entities on the Hindi gold set; covers Phase 3's fp16 scoring
TOP_EXAMPLES = 12  # how many example entities to print per side-by-side comparison table

# Phase 2 best.pt (epoch 4) from the public Drive folder linked in results.csv.
BILSTM_DRIVE_ID = "1r7RRTybH_j6Ir-3_SseX3iwoNmiGu-8H"  # Google Drive file ID for the frozen BiLSTM-CRF checkpoint from Phase 2
## Why hash-verify the checkpoint instead of trusting the Drive link: a shared Drive file can be silently replaced, partially downloaded, or corrupted without the URL changing; comparing against a pinned SHA-256 turns a silent wrong-checkpoint bug into a loud RuntimeError in download_bilstm(), before any score is computed from it.
BILSTM_SHA256 = "ca058bc9ca4519c366c1d0c72ab0148c51e98540039f49ad04550190bc64f291"  # expected hash of that checkpoint file, verified after download to guard against a corrupted or wrong download
BASE_TOKENIZER = "google/muril-base-cased"  # the tokenizer used for computing tokenizer-related stats (subword counts, [UNK] rate) later in the notebook

# Phase 2 architecture.
WORD_DIM = 300  # dimensionality of the pretrained word embeddings used by the BiLSTM-CRF
CHAR_DIM = 30  # dimensionality of the per-character embeddings (captures sub-word/morphological signal)
CHAR_HIDDEN = 25  # hidden size of the character-level BiLSTM (doubled to 50 once forward+backward are concatenated)
HIDDEN = 128  # hidden size of the main word-level BiLSTM (doubled to 256 once forward+backward are concatenated)
DROPOUT = 0.5  # dropout rate applied to embeddings/hidden states to reduce overfitting
MAX_WORD_CHARS = 20  # words are truncated to this many characters before being fed to the char-level encoder

In [4]:
import csv  # used to read results/results.csv (the shared scoreboard) as dict rows
import gc  # used to force garbage collection when freeing GPU memory between model loads
import glob  # general file-pattern matching utility available to this analysis section
import gzip  # used to write compressed .csv.gz / .jsonl.gz output files
import hashlib  # used to SHA-256 hash files: results.csv (read-only guard) and the downloaded BiLSTM checkpoint
import io  # used to wrap a gzip stream in a text layer so pandas/json can write to it as text
import json  # used to read/write JSON Lines prediction files and the run_info.json summary
import re  # used to validate/parse the pinned Hugging Face checkpoint URLs
import tempfile  # used to create a scratch output directory when running in DEBUG mode
import unicodedata  # used to classify characters (letter vs mark) when detecting an entity's script
from collections import Counter, defaultdict  # Counter for tallying; defaultdict for building nested prediction dicts
from importlib.metadata import version  # used to record installed package versions in the run's provenance info

import numpy as np  # numeric helpers: means and standard deviations for the transfer-gap statistics
import pandas as pd  # tabular data: every summary table in this notebook is a DataFrame
import torch  # the deep learning framework used to run both the BiLSTM-CRF and the transformer models

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 70)  # widen pandas' console display so the wide summary tables below don't get truncated

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # prefer GPU if one is attached, otherwise fall back to CPU
print("torch", torch.__version__, "| device:", device, "| visible GPUs:", torch.cuda.device_count())  # sanity-check print of the runtime environment
if not DEBUG and torch.cuda.device_count() != 1:  # the real run is only validated against exactly one visible GPU
## Why require exactly one GPU rather than letting Kaggle's dual-T4 sessions use both: nothing in this notebook explicitly places tensors/models on a specific device index beyond `torch.device("cuda")`, so with two visible GPUs PyTorch's default device choices could become ambiguous across cells; failing fast here is safer than silently running part of a model on the wrong GPU.
    raise RuntimeError(f"real runs need exactly one visible GPU, found {torch.cuda.device_count()}")  # fail fast rather than silently running on the wrong device setup

RESULTS_CSV = os.path.join(REPO_ROOT, "results", "results.csv")  # path to the shared, append-only scoreboard this notebook must not modify


def file_sha(path):  # small helper: SHA-256 of a file's raw bytes, used for both download verification and the read-only guard
    with open(path, "rb") as f:  # read the file in binary mode so hashing sees the raw bytes
        return hashlib.sha256(f.read()).hexdigest()  # hex digest string, easy to compare/print/log


## Why hash the file instead of simply never writing to it: a hash-before/hash-after check catches ANY accidental modification, including one introduced by a library call or a copy-paste mistake elsewhere in the notebook, not just the writes the author remembered to avoid - the final check near the end of the notebook re-verifies this same hash.
RESULTS_SHA = file_sha(RESULTS_CSV)  # snapshot the hash of results.csv now, so the end of the notebook can confirm it never changed
## Why route DEBUG output to a throwaway temp directory instead of results/error_analysis: DEBUG only scores 40 sentences per language as a smoke test, so its numbers are meaningless; writing into the same tracked folder as a real run would risk a later real run (or a careless git add) committing the fake, 40-sentence results as if they were the official analysis.
OUT_DIR = tempfile.mkdtemp(prefix="naamrekha_errors_debug_") if DEBUG else os.path.join(REPO_ROOT, "results", "error_analysis")  # DEBUG writes to a throwaway temp dir; a real run writes into the tracked results folder
os.makedirs(OUT_DIR, exist_ok=True)  # ensure the output directory exists before anything tries to write into it


def download_bilstm():  # fetches (or reuses a cached) Phase 2 BiLSTM-CRF checkpoint, verified by hash
    path = os.path.join(tempfile.gettempdir(), "naamrekha_phase2_best.pt")  # a fixed temp-dir path so repeated runs reuse the same cached download
    if not os.path.isfile(path) or file_sha(path) != BILSTM_SHA256:  # only download if missing or if the cached copy doesn't match the expected hash
        url = f"https://drive.usercontent.google.com/download?id={BILSTM_DRIVE_ID}&confirm=t"  # Google Drive direct-download URL; &confirm=t skips the "file is too large to scan" interstitial
        subprocess.run(["curl", "-L", "--fail", "--silent", "--show-error", "--retry", "3", "-o", path, url], check=True)  # download with redirects followed, failing loudly on HTTP errors, retrying transient failures
    if file_sha(path) != BILSTM_SHA256:  # re-check after downloading (or after finding a stale cached file)
        os.remove(path)  # delete the bad file so it isn't mistaken for a good cache next time
        raise RuntimeError("downloaded best.pt does not match the expected SHA-256")  # fail loudly rather than silently scoring with a wrong/corrupt checkpoint
    return path  # path to a verified-good checkpoint file


BILSTM_FILE = download_bilstm()  # actually perform the (possibly cached) download+verification
print("BiLSTM-CRF checkpoint:", BILSTM_FILE, "| SHA-256 verified")  # confirms to the reader that the checkpoint is present and trustworthy
print("outputs ->", OUT_DIR)  # confirms where this run's output files will be written

torch 2.11.0+cu130 | device: cuda | visible GPUs: 1
BiLSTM-CRF checkpoint: /tmp/naamrekha_phase2_best.pt | SHA-256 verified
outputs -> /kaggle/working/NaamRekha/results/error_analysis


## Data and run plan

Gold test sets are always the full shared splits, except in `DEBUG`.

In [5]:
gold_all = {lang: load_split(lang, "test") for lang in LANGS}  # load the full gold test split for each of the three languages (list of (words, tags) pairs)
gold = {lang: rows[:DEBUG_SENTENCES] if DEBUG else rows for lang, rows in gold_all.items()}  # DEBUG mode scores only a small prefix; a real run uses every sentence
print({lang: len(rows) for lang, rows in gold.items()})  # sanity-check print: how many sentences will actually be scored per language

with open(RESULTS_CSV, encoding="utf-8", newline="") as f:  # newline="" is required so csv's own universal-newline handling isn't double-applied
    SHEET = list(csv.DictReader(f))  # the whole shared scoreboard, as a list of {column: value} dicts, one per logged run


def parse_notes(row):  # the `notes` column packs extra metadata as "key=value; key=value; ..."
    return dict(part.split("=", 1) for part in row["notes"].split("; ") if "=" in part)  # split into a dict, ignoring any note fragment without an "="


## Why fail loudly on zero OR multiple matches instead of just taking found[0]: silently picking the first of several ambiguous rows (e.g. two seed-42 runs logged by mistake) would make the notebook reproduce an arbitrary, possibly wrong, score with no warning.
def only(found, what):  # small guard: expects exactly one matching row and fails loudly otherwise
    if len(found) != 1:  # zero or more than one match both indicate a data problem
        raise RuntimeError(f"expected one results row for {what}, found {len(found)}")  # ambiguous or missing row is a bug, not something to silently guess around
    return found[0]  # the single matching row


def phase4_row(train, test, seed):  # finds the one Phase 4 MuRIL results.csv row for a given train-language -> test-language -> seed combination
    return only([r for r in SHEET if r["model"] == "MuRIL" and "phase=4" in r["notes"] and r["train_lang"] == train  # first half of the filter: right model, right phase, right training language
                 and r["test_lang"] == test and parse_notes(r).get("seed") == str(seed)], f"MuRIL {train}->{test} s{seed}")  # filter the sheet down to that exact run and unwrap it


## Why require a pinned "tree/<40-hex-sha>" URL instead of accepting any checkpoint_path: a mutable reference (e.g. a branch name) could point to different weights at different times, silently breaking the "reproduce the logged F1" guarantee this whole reload step exists to provide.
def pinned(row):  # parses a results.csv checkpoint_path into (hf_repo, commit_sha), enforcing that it's pinned to an exact revision
    m = re.fullmatch(r"https://huggingface\.co/([^/]+/[^/]+)/tree/([0-9a-f]{40})", row["checkpoint_path"])  # expects the Hugging Face "tree/<40-hex-char-sha>" URL form
    if not m:  # the checkpoint_path string didn't match the expected pinned-URL pattern
        raise RuntimeError(f"not a pinned checkpoint: {row['checkpoint_path']}")  # refuse to proceed with an unpinned/mutable checkpoint reference
    return m.group(1), m.group(2)  # (e.g. "org/model-name", "abcdef0123...")


META, REF, BERT_RUNS = {}, {}, []  # META: per-system metadata; REF: (system, lang) -> logged F1 to reproduce; BERT_RUNS: transformer checkpoints to load


def add_bert(sid, family, train_lang, seed, refs):  # registers one transformer system (one checkpoint, possibly evaluated on several languages)
    repo, revision = pinned(next(iter(refs.values())))  # all languages for this system share the same checkpoint, so just read the pin from any one of them
    META[sid] = dict(family=family, train_lang=train_lang, seed=seed)  # record identifying metadata for later grouping/printing
    for lang, row in refs.items():  # record this system's logged F1 for every language it was evaluated on
        REF[(sid, lang)] = float(row["F1_overall"])  # the score this system must reproduce for that language
    BERT_RUNS.append(dict(sid=sid, repo=repo, revision=revision, langs=list(refs)))  # queue this checkpoint to actually be loaded and run


for seed in SEEDS:  # repeat the registration once per training seed (42, 43, 44)
    add_bert(f"MuRIL-hi-s{seed}", "MuRIL", "hi", seed, {t: phase4_row("hi", t, seed) for t in LANGS})  # Hindi-trained MuRIL, evaluated zero-shot on all three languages
    for lang in TARGET_LANGS:  # and once per target language (Marathi, Bengali) for the in-language models
        add_bert(f"MuRIL-{lang}-s{seed}", "MuRIL", lang, seed, {lang: phase4_row(lang, lang, seed)})  # the in-language MuRIL model for Marathi/Bengali, evaluated on its own language only

add_bert("IndicBERT", "IndicBERT", "hi", 42, {"hi": only([r for r in SHEET if r["model"] == "IndicBERT"], "IndicBERT")})  # the single IndicBERT run, Hindi only

META["BiLSTM-CRF"] = dict(family="BiLSTM-CRF", train_lang="hi", seed=42)  # the non-transformer baseline isn't loaded via BERT_RUNS, so register it separately
REF[("BiLSTM-CRF", "hi")] = float(only([r for r in SHEET if r["model"] == "BiLSTM-CRF"], "BiLSTM-CRF")["F1_overall"])  # its logged Hindi F1 to reproduce

LLM_SYSTEMS = {"LLM-zero": "LLM zero-shot [qwen2.5-7b-4bit]", "LLM-few": "LLM few-shot [qwen2.5-7b-4bit]"}  # maps short system ids to the exact results.csv model-name strings for the LLM runs
for sid, prefix in (("LLM-zero", "LLM-zero-shot ("), ("LLM-few", "LLM-few-shot (")):  # register both LLM prompting modes using their results.csv model-name prefixes
    META[sid] = dict(family="LLM", train_lang="none", seed=42)  # the LLM is prompted, not trained, hence train_lang="none"
    REF[(sid, "hi")] = float(only([r for r in SHEET if r["model"].startswith(prefix)], prefix)["F1_overall"])  # match by prefix since the full model name embeds the specific LLM used

print(f"{len(BERT_RUNS)} transformer checkpoints | {len(REF)} system-language scores to reproduce")  # summary of how much work the reproduction-check step below has to do

data/hi_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

data/hi_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

data/mr_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 25.2MB            

data/mr_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

data/bn_IndicNER_v1.0.zip: reconstructing file:   0%|          |  0.00B / 65.8MB            

data/bn_IndicNER_v1.0.zip: downloading bytes:           |  0.00B            

{'hi': 864, 'mr': 1060, 'bn': 607}
10 transformer checkpoints | 19 system-language scores to reproduce


## Prediction code

Transformer alignment and prediction are the same as Phase 4: labels are read from each word's first real subword, a
standalone `▁` is skipped, and truncated words count as `O`. Models are freshly loaded, so they run in full 32-bit.

In [6]:
def label_positions(word_ids, tokens):  # for each original word, find the single subword token position that should carry its predicted label
    # word -> position of the subword that carries its label: the first piece,
    # skipping a standalone SentencePiece "▁" when a real piece follows it.
    positions = {}  # word index -> subword position, filled in below
    for pos, wid in enumerate(word_ids):  # word_ids aligns each subword token back to the index of the original word it came from (None for special tokens)
## Why skip a standalone "▁" marker and prefer the next real subword: SentencePiece sometimes emits a bare "▁" (word-start marker) as its own token with no lexical content; letting that token carry the word's predicted label would mean classifying based on a token that encodes no information about the word itself.
        if wid is not None and (wid not in positions or tokens[positions[wid]] == "▁"):  # take the first real subword for this word, unless it was just a standalone "▁" marker
            positions[wid] = pos  # record/overwrite with this subword's position as the label-bearing one
    return positions  # dict: word index -> subword position holding that word's predicted tag


def predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH, batch_size=EVAL_BATCH_SIZE):  # runs a transformer token-classification model over pre-tokenized (word, gold-tags) rows
    model.eval()  # disable dropout etc. for deterministic inference
    dev = next(model.parameters()).device  # discover which device the model's weights live on, to move inputs there
    y_pred = []  # will hold one predicted tag sequence per input row
    for i in range(0, len(rows), batch_size):  # process the rows in batches for speed
        chunk = rows[i:i + batch_size]  # the current batch of (words, gold_tags) pairs
        enc = tokenizer([r[0] for r in chunk], is_split_into_words=True, truncation=True,  # tokenize this batch's word lists (first half of the call, continued below)
                        max_length=max_length, padding=True, return_tensors="pt").to(dev)  # subword-tokenize the already word-split sentences, pad to a common length, move to the model's device
        with torch.no_grad():  # no gradients needed for pure inference
            preds = model(**enc).logits.argmax(-1).cpu().numpy()  # forward pass, take the highest-scoring label id per subword position
        for b, (_, tags) in enumerate(chunk):  # walk each sentence in the batch to build its word-level predicted tags
            pred_tags = ["O"] * len(tags)  # default every word to "O"; words whose subword prediction we can't map stay "O"
            for wid, pos in label_positions(enc.word_ids(batch_index=b), enc.tokens(batch_index=b)).items():  # map each word back to its label-bearing subword position
                pred_tags[wid] = id2label[int(preds[b][pos])]  # translate that subword's predicted label id into its string tag (e.g. "B-PER")
            y_pred.append(pred_tags)  # store this sentence's full predicted tag sequence
    return y_pred  # list of predicted tag sequences, same order and length as `rows`

In [7]:
import torch.nn as nn  # neural network building blocks (Embedding, LSTM, Linear, Dropout)
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence  # utilities for running an LSTM efficiently over variable-length, padded sequences
from torchcrf import CRF  # linear-chain Conditional Random Field layer, used to decode globally-consistent tag sequences


class BiLSTMCRF(nn.Module):  # the frozen Phase 2 architecture, rebuilt here only to load and run its saved weights
    # Same architecture and parameter names as notebooks/phase2-bilstm-crf.ipynb.
    def __init__(self, n_words, n_chars, n_tags):  # n_words/n_chars/n_tags: vocabulary sizes loaded from the checkpoint, so embeddings match exactly what was trained
        super().__init__()  # standard nn.Module initialization
        self.word_emb = nn.Embedding(n_words, WORD_DIM, padding_idx=0)  # looks up a dense vector per word id; id 0 is the padding token and stays a zero vector
        self.char_emb = nn.Embedding(n_chars, CHAR_DIM, padding_idx=0)  # per-character embedding, feeding the char-level encoder below
        self.char_lstm = nn.LSTM(CHAR_DIM, CHAR_HIDDEN, batch_first=True, bidirectional=True)  # encodes each word's character sequence into a fixed-size vector (captures morphology/spelling)
        self.lstm = nn.LSTM(WORD_DIM + 2 * CHAR_HIDDEN, HIDDEN, batch_first=True, bidirectional=True)  # main sequence encoder over [word_emb ; char_features] per token
        self.dropout = nn.Dropout(DROPOUT)  # regularization applied to embeddings and LSTM outputs
        self.proj = nn.Linear(2 * HIDDEN, n_tags)  # projects the bidirectional LSTM's hidden state to one score per tag (CRF "emission" scores)
## Why a CRF layer instead of taking argmax over self.proj's per-token scores directly: a plain per-token argmax (as predict_tags() does for the transformer models) can produce impossible tag transitions like O -> I-PER with no preceding B-PER; the CRF's Viterbi decode (self.crf.decode below) scores whole tag SEQUENCES and enforces globally valid transitions.
        self.crf = CRF(n_tags, batch_first=True)  # learns tag-to-tag transition scores and decodes the best overall tag sequence (not just per-token argmax)

    def char_features(self, chars, char_lens):  # encodes each word's characters into a fixed-length vector via a small char-level BiLSTM
        B, T, L = chars.shape  # B=batch size, T=max sentence length (words), L=max word length (characters)
        flat, lens = chars.view(B * T, L), char_lens.view(-1)  # flatten (batch, word) into one axis so every word is processed as an independent sequence
        out = chars.new_zeros(B * T, 2 * CHAR_HIDDEN, dtype=torch.float)  # output buffer; padding "words" (length 0) stay all-zero
## Why mask out zero-length "words" before running the char-LSTM: packing a zero-length sequence into pack_padded_sequence would error out; computing char features only for `real` (actual) words and leaving `out` as zeros elsewhere avoids that crash while still producing a correctly-shaped batch.
        real = lens > 0  # mask selecting only the actual words (as opposed to padding positions with length 0)
        packed = pack_padded_sequence(self.char_emb(flat[real]), lens[real].cpu(),  # embed the real (non-padding) words' characters, then pack by true length (continued below)
                                      batch_first=True, enforce_sorted=False)  # pack so the LSTM skips computation on padding characters; enforce_sorted=False allows arbitrary input order
        _, (h, _) = self.char_lstm(packed)  # run the char-level BiLSTM; h holds the final hidden state from each direction
        out[real] = torch.cat([h[0], h[1]], dim=-1)  # concatenate forward+backward final states as this word's character-level feature vector
        return out.view(B, T, -1)  # reshape back to (batch, sentence_length, feature_dim)

    def decode(self, words, chars, char_lens, mask):  # full forward pass: embeddings -> BiLSTM -> CRF decode, returning the best tag sequence per sentence
        x = torch.cat([self.word_emb(words), self.char_features(chars, char_lens)], dim=-1)  # concatenate word-level and character-level representations per token
        x = self.dropout(x)  # regularize the combined input representation
        packed = pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)  # pack by true sentence lengths (mask.sum gives each sentence's real length) for efficient LSTM computation
        h, _ = self.lstm(packed)  # run the main bidirectional LSTM over the packed sequence
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=words.size(1))  # unpack back to a regular padded tensor matching the original sentence length
        return self.crf.decode(self.proj(self.dropout(h)), mask=mask)  # emission scores from the linear projection, then CRF Viterbi-decodes the globally best tag path per sentence


def bilstm_batch(batch, stoi, ctoi):  # turns a batch of word-lists into the padded word/char id tensors the model expects
    B, T = len(batch), max(len(w) for w in batch)  # B=batch size, T=longest sentence in this batch (in words)
    encoded = [[[ctoi.get(ch, 1) for ch in x[:MAX_WORD_CHARS]] for x in w] for w in batch]  # map each word's (truncated) characters to ids, with 1 as the "unknown character" id
    L = max(len(cw) for sent in encoded for cw in sent)  # longest word (in characters, after truncation) across the whole batch
    words = torch.zeros(B, T, dtype=torch.long)  # word-id tensor, padded with 0 ("pad" word id)
    chars = torch.zeros(B, T, L, dtype=torch.long)  # character-id tensor per word, padded with 0
    char_lens = torch.zeros(B, T, dtype=torch.long)  # actual character length of each word (0 for padding positions)
    mask = torch.zeros(B, T, dtype=torch.bool)  # True for real word positions, False for sentence-level padding
    for b, (sent, char_sent) in enumerate(zip(batch, encoded)):  # fill in each sentence's word ids, mask, and character tensors
        words[b, :len(sent)] = torch.tensor([stoi.get(x, 1) for x in sent])  # map each word to its vocabulary id (1 = unknown word)
        mask[b, :len(sent)] = True  # mark the real (non-padding) word positions for this sentence
        for j, cw in enumerate(char_sent):  # and each word's encoded character ids within that sentence
            chars[b, j, :len(cw)] = torch.tensor(cw)  # write this word's character ids into the padded character tensor
            char_lens[b, j] = len(cw)  # record this word's true character length
    return words, chars, char_lens, mask  # everything BiLSTMCRF.decode needs for this batch


def predict_bilstm(path, rows, dev, batch_size=256):  # loads the frozen BiLSTM-CRF checkpoint and predicts tags for a list of (words, gold_tags) rows
    ck = torch.load(path, map_location="cpu", weights_only=True)  # load the checkpoint dict (vocab, label order, and model weights) onto CPU first
## Why compare the checkpoint's own saved label order to LABELS: a model's predicted tag ids are meaningless without knowing which id maps to which string label; if the checkpoint were trained with a different label ordering, id2label[id] in predict_bilstm() would translate every prediction into the WRONG tag, silently corrupting every score instead of erroring.
    if list(ck["labels"]) != LABELS:  # guard: the checkpoint's own label order must match this project's shared label list
        raise RuntimeError("BiLSTM-CRF checkpoint label order differs from the shared label map")  # mismatched label order would silently corrupt predictions, so fail loudly instead
    stoi, ctoi = {w: i for i, w in enumerate(ck["itos"])}, {c: i for i, c in enumerate(ck["chars"])}  # rebuild word->id and char->id lookup tables from the checkpoint's saved vocabularies
    model = BiLSTMCRF(len(ck["itos"]), len(ck["chars"]), len(LABELS))  # construct a fresh model with vocab sizes matching this checkpoint
    model.load_state_dict(ck["model"])  # load the trained weights into it
    model.to(dev).eval()  # move to the target device and switch to inference mode
    out = []  # collected predicted tag sequences
    with torch.no_grad():  # no gradient tracking needed for inference
        for i in range(0, len(rows), batch_size):  # process the rows in batches, mirroring predict_tags above
            batch = [r[0] for r in rows[i:i + batch_size]]  # this batch's word lists (gold tags aren't needed for prediction)
            tensors = [t.to(dev) for t in bilstm_batch(batch, stoi, ctoi)]  # build and move the padded input tensors to the model's device
            out.extend([id2label[x] for x in seq] for seq in model.decode(*tensors))  # CRF-decode each sentence, translating predicted tag ids back to string labels
    del model  # drop the reference so Python can free the model's GPU/CPU memory promptly
    return out  # list of predicted tag sequences, same order as `rows`

## Reload every system and reproduce its logged score

The run stops here if any score differs from `results.csv` by more than 0.001.

In [8]:
from transformers import AutoModelForTokenClassification, AutoTokenizer  # Hugging Face classes used to load each pinned transformer checkpoint and its tokenizer

PREDS = {}  # (system_id, lang) -> list of predicted tag sequences, filled in by every system below


def free_memory():  # releases Python objects and clears the CUDA cache so the next model has room on the GPU
    gc.collect()  # force Python garbage collection of just-deleted model objects
    if torch.cuda.is_available():  # only clearing a CUDA cache makes sense if there is a GPU in the first place
        torch.cuda.empty_cache()  # return freed GPU memory to the CUDA allocator's free pool


PREDS[("BiLSTM-CRF", "hi")] = predict_bilstm(BILSTM_FILE, gold["hi"], device)  # score the BiLSTM-CRF baseline on the Hindi gold set (its only evaluated language here)
free_memory()  # clear GPU memory before moving on to the transformer models

for run in BERT_RUNS:  # one iteration per registered MuRIL/IndicBERT checkpoint
    tok = AutoTokenizer.from_pretrained(run["repo"], revision=run["revision"])  # load the tokenizer pinned to the exact commit the checkpoint was saved at
    model = AutoModelForTokenClassification.from_pretrained(run["repo"], revision=run["revision"]).to(device)  # load the token-classification model at that same pinned revision
## Same reasoning as the BiLSTM-CRF label-order guard above: a transformer checkpoint with its own differently-ordered label2id would make `id2label[int(preds[b][pos])]` in predict_tags() silently return the wrong tag string for every prediction.
    if model.config.label2id != label2id:  # guard: the checkpoint's own label mapping must match this project's shared mapping
        raise RuntimeError(f"{run['sid']}: checkpoint label map differs from src.label2id")  # a mismatched label map would silently scramble predicted tag ids
    for lang in run["langs"]:  # a Hindi-trained model is scored on all three languages; an in-language model only on its own
        PREDS[(run["sid"], lang)] = predict_tags(model, tok, gold[lang])  # run prediction and store results keyed by (system id, language)
    print("scored", run["sid"], run["langs"])  # progress print so a long reload loop is visible
    del model, tok  # drop references to let memory be freed
    free_memory()  # clear GPU memory before loading the next checkpoint

saved = defaultdict(dict)  # system_name -> {sentence index: saved prediction record}, used to recover the committed LLM predictions
with open(os.path.join(REPO_ROOT, "results", "phase5", "final_predictions.jsonl"), encoding="utf-8") as f:  # the LLM is not re-run here; its frozen Phase 5 predictions are read back instead
    for line in f:  # one saved prediction record per line
        rec = json.loads(line)  # one JSON record per (system, sentence)
        if rec["system"] in LLM_SYSTEMS.values():  # only keep records for the two LLM systems this notebook compares
            saved[rec["system"]][rec["idx"]] = rec  # index by sentence position for easy reordering below
for sid, name in LLM_SYSTEMS.items():  # reassemble each LLM system's predictions in canonical sentence order
    recs = [saved[name][i] for i in range(len(gold_all["hi"]))]  # reorder the saved records back into the canonical sentence order
## Why re-check the saved LLM predictions against today's gold split word-for-word: the LLM is never re-run here (its Phase 5 predictions are frozen/committed), so if the gold test split were ever regenerated or reordered, comparing frozen predictions against a silently different gold sequence would misattribute errors to the wrong sentences.
    if any(r["words"] != w or r["gold"] != t for r, (w, t) in zip(recs, gold_all["hi"])):  # guard: the saved predictions must line up with today's Hindi gold split, word-for-word
        raise RuntimeError(f"{sid}: saved words or gold differ from the shared Hindi test split")  # a mismatch would mean comparing predictions against the wrong sentences
    PREDS[(sid, "hi")] = [r["pred"] for r in recs[:len(gold["hi"])]]  # slice to DEBUG_SENTENCES if DEBUG, else keep all 864

score_rows = []  # will become the SCORES DataFrame: one row per (system, language) with its reproduced metrics
for (sid, lang), pred in PREDS.items():  # score every system-language pair that was just predicted or recovered
    s = evaluate([t for _, t in gold[lang]], pred)  # seqeval entity-level scoring: predicted tags vs this language's gold tags
    score_rows.append(dict(system=sid, lang=lang, **{k: round(v, 6) for k, v in s.items()}, logged=REF[(sid, lang)],  # build this row's dict (continued below): computed metrics plus the logged reference score
                           diff=s["F1_overall"] - REF[(sid, lang)]))  # keep the freshly computed scores alongside the originally logged F1 and their difference
SCORES = pd.DataFrame(score_rows).sort_values(["lang", "system"]).reset_index(drop=True)  # tidy, sorted table of every system-language score
print(SCORES.round(4).to_string(index=False))  # print the full reproduction table for manual inspection
if len(PREDS) != len(REF):  # guard: every expected (system, language) pair from REF must have actually been scored
    raise RuntimeError(f"scored {len(PREDS)} system-language pairs, expected {len(REF)}")  # catches an accidentally skipped system/language combination
## Why stop the whole notebook (below) rather than just flagging these rows: every error-analysis table that follows assumes each system's predictions match its officially logged score; if a score didn't reproduce, the entity-level errors counted from it could not be trusted as representative of that system's real, logged behavior.
bad = SCORES[SCORES["diff"].abs() > REPRO_TOLERANCE]  # rows where today's score drifted from the logged one by more than the allowed tolerance
if DEBUG:  # a DEBUG slice is too small to expect the full-dataset scores to reproduce
    print("DEBUG: scores use a tiny slice, so reproduction is not checked.")  # a 40-sentence slice is expected to score differently from the full 864-sentence logged run
elif len(bad):  # a real run where at least one system-language score drifted beyond tolerance
    raise RuntimeError("logged scores not reproduced:\n" + bad.to_string(index=False))  # stop the notebook: error analysis on unreproducible scores would be untrustworthy
else:  # a real run where every system-language score reproduced within tolerance
    print(f"all {len(SCORES)} scores reproduce within {REPRO_TOLERANCE}")  # confirms every system's score matches its logged value closely enough to trust the error analysis that follows

config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  948MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (61 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  131MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

scored IndicBERT ['hi']
      system lang  F1_overall  F1_PER  F1_ORG  F1_LOC  logged    diff
MuRIL-bn-s42   bn      0.8136  0.8621  0.7233  0.8018  0.8136  0.0000
MuRIL-bn-s43   bn      0.8100  0.8492  0.7373  0.8000  0.8100  0.0000
MuRIL-bn-s44   bn      0.8159  0.8642  0.7427  0.7932  0.8159  0.0000
MuRIL-hi-s42   bn      0.7982  0.8506  0.6963  0.7906  0.7982  0.0000
MuRIL-hi-s43   bn      0.8036  0.8487  0.7246  0.7902  0.8036  0.0000
MuRIL-hi-s44   bn      0.8091  0.8522  0.7330  0.7962  0.8091  0.0000
  BiLSTM-CRF   hi      0.8038  0.8692  0.7130  0.7933  0.8038  0.0000
   IndicBERT   hi      0.7831  0.8538  0.6949  0.7686  0.7839 -0.0007
     LLM-few   hi      0.6643  0.6714  0.5956  0.7190  0.6643  0.0000
    LLM-zero   hi      0.6328  0.6397  0.5644  0.6812  0.6328  0.0000
MuRIL-hi-s42   hi      0.8274  0.8879  0.7444  0.8224  0.8274  0.0000
MuRIL-hi-s43   hi      0.8237  0.8907  0.7327  0.8187  0.8237  0.0000
MuRIL-hi-s44   hi      0.8174  0.8816  0.7197  0.8218  0.8174  0.0

## Error categories

These are the Phase 5 categories, so counts stay comparable:

- `missed`: no predicted span touches the gold entity.
- `boundary`: a predicted span overlaps it with different edges.
- `type_confusion`: exact span, wrong type.
- `spurious`: a predicted span touches no gold entity.

If one gold entity overlaps several predicted fragments, it still counts as one boundary error. `n_overlaps` records how
many fragments there were.

In [9]:
from seqeval.metrics.sequence_labeling import get_entities  # turns a flat IOB2 tag sequence into a list of (type, start, end) entity spans

KINDS = ["boundary", "missed", "spurious", "type_confusion"]  # the four mutually-exclusive error categories tracked in this notebook
TYPES = ["PER", "ORG", "LOC"]  # the three entity types this project tags
SCRIPT_BLOCKS = {"Devanagari": (0x0900, 0x097F), "Bengali": (0x0980, 0x09FF)}  # Unicode code-point ranges used to detect which script a character belongs to


def spans(tags):  # convenience wrapper: get_entities returns (type, start, end) tuples; reorder to (start, end, type)
    return [(s, e, t) for t, s, e in get_entities(tags)]  # reorder seqeval's (type, start, end) into (start, end, type) for easier sorting/comparison


def classify(words, gold_tags, pred_tags):  # compares one sentence's gold vs predicted entities and labels each discrepancy with an error kind
    G, P = spans(gold_tags), spans(pred_tags)  # gold and predicted entity spans for this sentence
    gset, pset = set(G), set(P)  # sets for fast exact-match membership tests
    text = lambda sp: " ".join(words[sp[0]:sp[1] + 1])  # recovers the surface text of a span from the word list
    out = []  # accumulates one error record per mismatched gold or spurious predicted span
    for g in G:  # check every gold entity against what was predicted
        if g in pset:  # exact span and type match: no error to record
            continue  # this gold entity was predicted exactly right, so it's not an error
## Why check `same` (type_confusion) before `over` (boundary): an exact-boundary-but-wrong-type prediction also technically "overlaps" the gold span, so checking `over` first would misclassify every type_confusion as a boundary error, hiding how often the model gets the SPAN right but the TYPE wrong.
        same = [p for p in P if (p[0], p[1]) == (g[0], g[1])]  # predicted spans with identical boundaries (so if present, only the type can be wrong)
        over = [p for p in P if (p[0], p[1]) != (g[0], g[1]) and not (p[1] < g[0] or p[0] > g[1])]  # predicted spans that overlap this gold entity but with different boundaries
        kind, p = ("type_confusion", same[0]) if same else ("boundary", over[0]) if over else ("missed", None)  # pick the matching error category and, if any, the offending prediction
        out.append(dict(kind=kind, gold=text(g), gold_type=g[2], pred=text(p) if p else "",  # record this gold entity's error (continued below): what was predicted instead, if anything
## Why record n_overlaps instead of only the error kind: one gold entity can overlap several predicted fragments (the model split it into pieces); without this count, a single badly-fragmented entity and a single cleanly-shifted-boundary entity would both just read "boundary", losing the distinction between the two failure modes.
                        pred_type=p[2] if p else "", n_overlaps=len(over)))  # record both the gold and (if any) predicted text/type for this error
    for p in P:  # separately, find predicted spans that don't correspond to any gold entity at all
        if p not in gset and not any(not (g[1] < p[0] or g[0] > p[1]) for g in G):  # not an exact match and doesn't even overlap any gold entity
            out.append(dict(kind="spurious", gold="", gold_type="", pred=text(p), pred_type=p[2], n_overlaps=0))  # a prediction with nothing gold behind it at all
    return out  # list of per-entity error records for this sentence


## Why classify scripts with raw Unicode code-point ranges instead of a language-detection library: a general-purpose detector is built for whole sentences, not single short entity strings, and would be nondeterministic/offline-unfriendly here; fixed code-point ranges give an exact, reproducible answer for exactly the two scripts (Devanagari, Bengali) this analysis cares about.
def char_script(ch):  # classifies a single character's script by Unicode code-point range
    code_point = ord(ch)  # the character's Unicode code point, compared against the script ranges below
    for name, (lo, hi) in SCRIPT_BLOCKS.items():  # check each known script's code-point range in turn
        if lo <= code_point <= hi:  # this character falls inside that script's Unicode block
            return name  # Devanagari or Bengali, based on which block the code point falls in
    if ch.isascii() and ch.isalpha():  # not Devanagari or Bengali, but still a plain Latin letter
        return "Latin"  # plain ASCII letters (e.g. an English name written in Hindi text)
    return "Other"  # punctuation, digits, or any other script not explicitly handled


def entity_script(text, lang):  # determines whether an entity's surface text is purely in its language's native script, purely Latin, or a mix
## Why restrict to Unicode categories starting with "L" (letter) or "M" (mark) instead of scanning every character: spaces, digits and punctuation are script-neutral and present in entities of every language, so including them would make an all-Devanagari-plus-spaces entity look "mixed_or_other" purely because of characters that carry no script information at all.
    scripts = {char_script(ch) for ch in text if unicodedata.category(ch)[0] in ("L", "M")}  # collect scripts only from letters/marks (category starting with L or M), ignoring spaces/punctuation
    if not scripts:  # no alphabetic characters were found in this entity's text at all
        return "no_letters"  # entity text has no alphabetic characters at all (unusual edge case)
    if scripts == {NATIVE_SCRIPT[lang]}:  # every letter belongs to this language's own native script
        return "native"  # every letter is in this language's expected native script
    if scripts == {"Latin"}:  # every letter is plain Latin script
        return "latin"  # the whole entity is written in Latin script (e.g. an untransliterated English name)
    return "mixed_or_other"  # a mix of scripts, or a script outside the two checked for


def subword_bucket(n):  # groups a raw subword count into coarse buckets for the recall-vs-tokenization-cost table
    return "1" if n <= 1 else "2" if n == 2 else "3-4" if n <= 4 else "5+"  # coarse buckets keep the recall table readable instead of one row per exact count


def entity_table(lang, sids):  # builds the per-entity correctness table used throughout the rest of the notebook
    # One row per gold entity, with one correct/incorrect column per system.
    records = []  # will hold one dict per gold entity, plus one boolean column per system in `sids`
    predicted = {sid: PREDS[(sid, lang)] for sid in sids}  # pre-fetch each requested system's predictions for this language
    for i, (words, tags) in enumerate(gold[lang]):  # walk every gold sentence for this language
        for s, e, t in spans(tags):  # every gold entity span in this sentence
            rec = dict(lang=lang, idx=i, start=s, end=e, type=t, text=" ".join(words[s:e + 1]))  # base record identifying this gold entity
            records.append(rec)  # add this gold entity's base record before filling in per-system correctness below
        found = {sid: set(spans(predicted[sid][i])) for sid in sids}  # this system's full set of predicted spans for this same sentence
        for rec in records[-len(spans(tags)):] if spans(tags) else []:  # revisit just the gold entities added for this sentence
            key = (rec["start"], rec["end"], rec["type"])  # the exact (start, end, type) tuple that must appear in a system's predictions to count as "correct"
            for sid in sids:  # check this entity against every requested system's predictions
                rec[sid] = key in found[sid]  # True/False: did this system predict this exact gold entity correctly
    return pd.DataFrame(records, columns=["lang", "idx", "start", "end", "type", "text", *sids])  # tidy table: one row per gold entity, one boolean column per system

In [10]:
err_rows = []  # flat list of per-entity error records across every (system, language) pair, feeding the ERR DataFrame
for (sid, lang), pred in PREDS.items():  # classify errors for every scored system-language pair
    for i, ((words, tags), p) in enumerate(zip(gold[lang], pred)):  # pair each gold sentence with this system's prediction for it
        for e in classify(words, tags, p):  # one or more error records for this sentence (possibly none, if everything was correct)
            err_rows.append(dict(system=sid, lang=lang, idx=i, **e))  # tag each error record with which system/language/sentence it came from
ERR = pd.DataFrame(err_rows, columns=["system", "lang", "idx", "kind", "gold", "gold_type", "pred", "pred_type", "n_overlaps"])  # the full per-error table used by every analysis section below
ERR["type"] = ERR["gold_type"].where(ERR["gold_type"] != "", ERR["pred_type"])  # for spurious errors (no gold_type), fall back to the predicted type so "type" is always meaningful

COUNTS = ERR.groupby(["lang", "system", "kind"]).size().unstack(fill_value=0).reindex(columns=KINDS, fill_value=0)  # error counts per (language, system), one column per error kind
COUNTS["total"] = COUNTS.sum(axis=1)  # overall error count per (language, system), across all kinds
TYPE_COUNTS = ERR.groupby(["lang", "system", "type", "kind"]).size().unstack(fill_value=0).reindex(columns=KINDS, fill_value=0)  # same breakdown, further split by entity type
TYPE_COUNTS["total"] = TYPE_COUNTS.sum(axis=1)  # overall error count per (language, system, type)

# The same categories must reproduce the committed Phase 5 error table.
## Why hardcode these exact counts and compare against them, instead of just printing fresh counts every run: these numbers are already written into docs/phase5/error_analysis.md and the Phase 6 report; this check turns any future accidental change to classify()'s logic (e.g. reordering the same/over check above) into an immediate RuntimeError instead of a silent, undetected drift from what was published.
PHASE5_COUNTS = {"LLM-zero": (293, 299, 478, 52), "LLM-few": (317, 227, 406, 53), "MuRIL-hi-s42": (167, 77, 205, 31)}  # (boundary, missed, spurious, type_confusion) counts previously committed for these three systems on Hindi
if not DEBUG:  # the committed counts were captured from a full, non-DEBUG run
    for sid, expected in PHASE5_COUNTS.items():  # check each of the three previously-committed systems in turn
        got = tuple(int(COUNTS.loc[("hi", sid), k]) for k in KINDS)  # this run's freshly computed counts for the same system/kinds
        if got != expected:  # the freshly computed counts no longer match what was committed
            raise RuntimeError(f"{sid}: Phase 5 error counts not reproduced: {got} vs {expected}")  # a mismatch would mean the error-classification logic changed or drifted
    print("Phase 5 error counts reproduced for LLM zero-shot, LLM few-shot, and MuRIL")  # confirms the error-counting pipeline itself is trustworthy before drawing conclusions from it

Phase 5 error counts reproduced for LLM zero-shot, LLM few-shot, and MuRIL


## 1. Hindi across stages

All systems use the same 864 sentences. MuRIL is the seed 42 handoff model.

In [11]:
HI_SYSTEMS = ["BiLSTM-CRF", "MuRIL-hi-s42", "IndicBERT", "LLM-zero", "LLM-few"]  # the five systems compared head-to-head on Hindi, in a fixed display order
print("Errors by kind:")  # section header for the kind-breakdown table that follows
print(COUNTS.loc["hi"].reindex(HI_SYSTEMS).to_string())  # error-kind breakdown for each of the five systems, in the chosen order
print("\nErrors by type (gold type, or predicted type for spurious spans):")  # section header for the type-breakdown table that follows
print(TYPE_COUNTS.loc["hi"].reindex(pd.MultiIndex.from_product([HI_SYSTEMS, TYPES]), fill_value=0).to_string())  # same breakdown, further split by PER/ORG/LOC

AGREE_SYSTEMS = ["BiLSTM-CRF", "MuRIL-hi-s42", "IndicBERT", "LLM-few"]  # the systems compared pairwise below (LLM zero-shot excluded to focus on the stronger, more comparable systems)
HI_ENT = entity_table("hi", AGREE_SYSTEMS)  # one row per Hindi gold entity, with a correct/incorrect column for each of these four systems
HI_ENT["n_correct"] = HI_ENT[AGREE_SYSTEMS].sum(axis=1)  # how many of the four systems got this particular entity right
print(f"\nGold entities by number of systems correct (out of {len(AGREE_SYSTEMS)}):")  # section header for the agreement cross-tab that follows
print(pd.crosstab(HI_ENT["type"], HI_ENT["n_correct"], margins=True).to_string())  # cross-tab: entity type vs. how many systems agreed on it, with row/column totals


def top_entities(frame, n=TOP_EXAMPLES):  # formats the most frequent entity texts in a filtered subset of HI_ENT, for quick qualitative inspection
    if frame.empty:  # guard against an empty filter result before grouping
        return "none"  # avoid crashing/printing nothing useful when a filter matches zero entities
    c = frame.groupby(["type", "text"]).size().sort_values(ascending=False).head(n)  # most common (type, text) combinations in this subset
    return ", ".join(f"{t}:{x} ({k})" for (t, x), k in c.items())  # human-readable "TYPE:text (count)" list


print("\nWrong in every system:", top_entities(HI_ENT[HI_ENT["n_correct"] == 0]))  # entities none of the four systems could get right - likely genuinely hard or ambiguous
print("\nMuRIL right, BiLSTM-CRF wrong:", top_entities(HI_ENT[HI_ENT["MuRIL-hi-s42"] & ~HI_ENT["BiLSTM-CRF"]]))  # entities where the transformer succeeds but the simpler BiLSTM-CRF fails
print("\nBiLSTM-CRF right, MuRIL wrong:", top_entities(HI_ENT[HI_ENT["BiLSTM-CRF"] & ~HI_ENT["MuRIL-hi-s42"]]))  # the rarer reverse case: where the simpler model outperforms MuRIL
print("\nMuRIL right, IndicBERT wrong:", top_entities(HI_ENT[HI_ENT["MuRIL-hi-s42"] & ~HI_ENT["IndicBERT"]]))  # compares the two transformer checkpoints against each other
print("\nMuRIL right, LLM few-shot wrong:", top_entities(HI_ENT[HI_ENT["MuRIL-hi-s42"] & ~HI_ENT["LLM-few"]]))  # cases where fine-tuning beats prompting
print("\nLLM few-shot right, MuRIL wrong:", top_entities(HI_ENT[HI_ENT["LLM-few"] & ~HI_ENT["MuRIL-hi-s42"]]))  # the reverse: cases where prompting beats fine-tuning

Errors by kind:
kind          boundary  missed  spurious  type_confusion  total
system                                                         
BiLSTM-CRF         173     197       148              33    551
MuRIL-hi-s42       167      77       205              31    480
IndicBERT          204     160       204              32    600
LLM-zero           293     299       478              52   1122
LLM-few            317     227       406              53   1003

Errors by type (gold type, or predicted type for spurious spans):
kind              boundary  missed  spurious  type_confusion  total
BiLSTM-CRF   PER        51      41        45               7    144
             ORG        62      89        65              15    231
             LOC        60      67        38              11    176
MuRIL-hi-s42 PER        49      10        58               6    123
             ORG        64      37        94               9    204
             LOC        54      30        53              16 

## 2. Zero-shot vs in-language on Marathi and Bengali

Differences are paired by seed: `in-language errors - zero-shot errors`. A **negative** value means the in-language model
makes fewer errors. `same_sign` shows whether all three seeds agree. Seeds measure training variation, not test-set
sampling variation.

"Consistent wins" are gold entities that all three models on one side get right and all three on the other side get wrong.

In [12]:
def error_count(sid, lang, kind=None, etype=None):  # looks up an error count from COUNTS or TYPE_COUNTS, defaulting to 0 if the combination has no rows at all
    if etype is None:  # overall (not type-specific) lookup, from the COUNTS table
        if (lang, sid) not in COUNTS.index:      # a tiny DEBUG slice can have no errors
            return 0  # avoids a KeyError when a DEBUG run happens to have zero errors for some system/language
        return int(COUNTS.loc[(lang, sid), "total" if kind is None else kind])  # overall or per-kind error count, not split by entity type
    key = (lang, sid, etype)  # type-specific lookup key into TYPE_COUNTS
    if key not in TYPE_COUNTS.index:  # this system/language/type combination had zero recorded errors
        return 0  # same zero-error edge case, but for the type-specific lookup
    return int(TYPE_COUNTS.loc[key, "total" if kind is None else kind])  # overall or per-kind error count, split by entity type


gap_rows = []  # will become the TRANSFER_GAPS DataFrame comparing zero-shot vs in-language MuRIL error counts
for lang in TARGET_LANGS:  # Marathi and Bengali: the two languages with both a zero-shot (Hindi-trained) and an in-language model
    for etype in [None, *TYPES]:  # compute the comparison overall (None) and separately per entity type
        for kind in [None, *KINDS]:  # and overall (None) as well as per error kind
            zero = [error_count(f"MuRIL-hi-s{s}", lang, kind, etype) for s in SEEDS]  # Hindi-trained model's error count, one value per seed
            own = [error_count(f"MuRIL-{lang}-s{s}", lang, kind, etype) for s in SEEDS]  # in-language model's error count, one value per seed
## Why "in-language minus zero-shot" and not the reverse: this fixes a convention (negative = in-language model makes fewer errors = transfer-to-in-language HELPS) that every later reader of diff_mean and same_sign can rely on without re-deriving it; flipping the subtraction order would silently invert the meaning of every negative number printed below.
            delta = [b - a for a, b in zip(zero, own)]  # per-seed difference: in-language errors minus zero-shot errors (negative = in-language model is better)
            gap_rows.append(dict(lang=lang, type=etype or "ALL", kind=kind or "total", zero_shot_mean=np.mean(zero),  # build this comparison row (continued below): means, per-seed deltas, and whether all seeds agree
                                 in_language_mean=np.mean(own), diff_mean=np.mean(delta),  # (continued) average error counts and their mean difference
                                 diff_sd=np.std(delta, ddof=1) if not DEBUG or len(delta) > 1 else 0.0,  # (continued) spread of the per-seed difference, to judge how noisy the effect is
                                 per_seed="/".join(f"{d:+d}" for d in delta),  # (continued) the raw per-seed deltas, formatted with explicit +/- signs
## Why check agreement across all three seeds instead of just looking at diff_mean: a mean difference can look like a real effect even when it is driven by one unusually lucky or unlucky seed; same_sign specifically asks whether every seed points the same direction, which is what separates a robust effect from noise.
                                 same_sign=all(d > 0 for d in delta) or all(d < 0 for d in delta)))  # same_sign: do all three seeds agree on the direction of the effect (not just its average)
TRANSFER_GAPS = pd.DataFrame(gap_rows)  # the full zero-shot vs in-language comparison, ready to print
print(TRANSFER_GAPS.round(2).to_string(index=False))  # full table of zero-shot vs in-language error gaps, by language/type/kind

win_rows, TRANSFER_ENT = [], {}  # win_rows -> CONSISTENT_WINS summary; TRANSFER_ENT -> per-language entity-level correctness tables, kept for later cells
for lang in TARGET_LANGS:  # find consistent per-entity wins for each target language in turn
    zero_sids = [f"MuRIL-hi-s{s}" for s in SEEDS]  # the three zero-shot (Hindi-trained) model runs
    own_sids = [f"MuRIL-{lang}-s{s}" for s in SEEDS]  # the three in-language model runs for this target language
    ent = entity_table(lang, zero_sids + own_sids)  # per-entity correctness for all six runs (three zero-shot + three in-language)
    TRANSFER_ENT[lang] = ent  # saved for reuse in the later script/tokenizer-effects section
    zero_wins = ent[ent[zero_sids].all(axis=1) & ~ent[own_sids].any(axis=1)]  # entities all three zero-shot seeds get right, and all three in-language seeds get wrong
    own_wins = ent[ent[own_sids].all(axis=1) & ~ent[zero_sids].any(axis=1)]  # the reverse: consistently right in-language, consistently wrong zero-shot
    for etype in [*TYPES, "ALL"]:  # break the win counts down by entity type, plus an overall total
        z = zero_wins if etype == "ALL" else zero_wins[zero_wins["type"] == etype]  # restrict to one entity type, or keep all
        o = own_wins if etype == "ALL" else own_wins[own_wins["type"] == etype]  # same restriction, for the in-language model's consistent wins
        total = len(ent) if etype == "ALL" else int((ent["type"] == etype).sum())  # total gold entities of this type, for context
        win_rows.append(dict(lang=lang, type=etype, gold_entities=total, zero_shot_wins=len(z), in_language_wins=len(o)))  # one summary row per language/type combination
    print(f"\n{lang}: Hindi models consistently right, in-language consistently wrong:", top_entities(zero_wins))  # qualitative examples of where more training data (in-language) actually hurts
    print(f"{lang}: in-language consistently right, Hindi models consistently wrong:", top_entities(own_wins))  # and the reverse: clear wins from in-language training
CONSISTENT_WINS = pd.DataFrame(win_rows)  # the full consistent-wins summary, ready to print
print("\nConsistent wins:")  # section header for the table that follows
print(CONSISTENT_WINS.to_string(index=False))  # summary table: how many entities are "consistent wins" for each side, by language and type

lang type           kind  zero_shot_mean  in_language_mean  diff_mean  diff_sd    per_seed  same_sign
  mr  ALL          total          393.00            435.33      42.33     5.03 +37/+47/+43       True
  mr  ALL       boundary          159.67            166.67       7.00     5.57   +6/+13/+2       True
  mr  ALL         missed           82.67            128.33      45.67     2.31 +47/+43/+47       True
  mr  ALL       spurious          125.67            113.33     -12.33     3.51  -16/-9/-12       True
  mr  ALL type_confusion           25.00             27.00       2.00     3.46    +0/+0/+6      False
  mr  PER          total           98.67            114.33      15.67     3.21 +12/+18/+17       True
  mr  PER       boundary           42.67             43.67       1.00     1.00    +2/+0/+1      False
  mr  PER         missed           16.00             38.67      22.67     4.04 +19/+27/+22       True
  mr  PER       spurious           38.00             28.67      -9.33     1.53   -

## 3. Script and tokenizer effects

This section tests the roadmap's "Bengali script tokens?" question in three ways: the script of each gold entity,
`[UNK]` subwords, and how many MuRIL subwords an entity needs. Recall is averaged across the three seeds.

Different languages use different test sets, so compare patterns, not absolute values across languages.

In [13]:
## Why load a tokenizer here purely to measure statistics, never to predict: this cell asks "is the MuRIL tokenizer itself part of why errors happen" (subword counts, [UNK] rate), which is a property of the tokenizer alone - it has to be measured independently of any trained model's predictions to be a fair explanatory variable for those predictions.
base_tok = AutoTokenizer.from_pretrained(BASE_TOKENIZER)  # the shared MuRIL tokenizer, used purely to measure subword/[UNK] statistics (not for prediction here)
word_cache = {}  # memoizes per-word tokenizer stats, since many words repeat across sentences


def word_info(word):  # returns (number of subwords, whether tokenization produced an [UNK]) for a single word, cached
    if word not in word_cache:  # only tokenize a word the first time it's seen
        ids = base_tok(word, add_special_tokens=False)["input_ids"]  # subword token ids for this word alone, without [CLS]/[SEP]
        word_cache[word] = (len(ids), base_tok.unk_token_id in ids)  # how many subwords it split into, and whether any of them was the unknown-token id
    return word_cache[word]  # cached (or freshly computed) (subword_count, has_unk) pair


token_rows = []  # will become the TOKEN_STATS DataFrame: per-language tokenizer statistics
for lang in LANGS:  # compute tokenizer statistics separately for each language
    info = [word_info(w) for words, _ in gold[lang] for w in words]  # (subword_count, has_unk) for every word in every gold sentence of this language
    token_rows.append(dict(lang=lang, words=len(info), mean_subwords=np.mean([n for n, _ in info]),  # one summary row per language (continued below): word count and average subwords per word
                           pct_words_multi_subword=100 * np.mean([n > 1 for n, _ in info]),  # (continued) percentage of words needing more than one subword
                           pct_words_with_unk=100 * np.mean([u for _, u in info])))  # average subwords per word, and the percentage of words that split into >1 subword or hit an [UNK]
TOKEN_STATS = pd.DataFrame(token_rows)  # the full per-language tokenizer-difficulty table, ready to print
print(TOKEN_STATS.round(3).to_string(index=False))  # per-language tokenizer-difficulty summary (more subwords/[UNK]s -> a harder-to-tokenize language for this tokenizer)

SCRIPT_ENT, recall_rows = {}, []  # SCRIPT_ENT -> per-language entity tables annotated with script/subword/unk features; recall_rows -> the SCRIPT_RECALL summary
for lang in LANGS:  # compute script/subword/[UNK] recall breakdowns separately for each language
    zero_sids = [f"MuRIL-hi-s{s}" for s in SEEDS]  # the Hindi-trained model's three seed runs, evaluated on this language
    own_sids = [] if lang == "hi" else [f"MuRIL-{lang}-s{s}" for s in SEEDS]  # the in-language model's runs, only applicable for Marathi/Bengali (Hindi has no separate "in-language" model here)
    ent = entity_table(lang, zero_sids + own_sids).copy()  # per-entity correctness for every relevant system, copied so new columns can be added safely
    infos = [[word_info(w) for w in gold[lang][r.idx][0][r.start:r.end + 1]] for r in ent.itertuples()]  # per-word tokenizer stats for just the words making up each gold entity
    ent["script"] = [entity_script(t, lang) for t in ent["text"]]  # native / latin / mixed_or_other / no_letters, per entity
    ent["subwords"] = [subword_bucket(sum(n for n, _ in x)) for x in infos]  # bucketed total subword count needed to tokenize this entity
    ent["has_unk"] = ["unk" if any(u for _, u in x) else "no_unk" for x in infos]  # whether tokenizing this entity produced at least one [UNK]
    SCRIPT_ENT[lang] = ent  # kept in case later analysis/debugging needs the per-entity detail
    groups = {"hindi_model": zero_sids}  # recall is always computed for the Hindi-trained model...
    if own_sids:  # only Marathi and Bengali have a separate in-language model to compare against
        groups["in_language"] = own_sids  # ...and, where available, for the in-language model too
    for factor in ("script", "subwords", "has_unk"):  # compute recall broken down by each of the three tokenizer-related factors in turn
        for level, frame in ent.groupby(factor):  # e.g. factor="script", level="native"/"latin"/...
            row = dict(lang=lang, factor=factor, level=level, entities=len(frame))  # base row identifying this language/factor/level combination
            for name, sids in groups.items():  # add a recall column for each model group being compared
                row[f"{name}_recall"] = frame[sids].to_numpy().mean() if len(frame) else np.nan  # fraction of entities at this factor level that the given model group got right
            recall_rows.append(row)  # add this fully-filled-in row to the summary table
SCRIPT_RECALL = pd.DataFrame(recall_rows)  # the full script/tokenizer-effects recall table, ready to print
print(SCRIPT_RECALL.round(4).to_string(index=False))  # full table: recall by language, factor (script/subword-count/unk), and level - the direct answer to "do tokenization issues explain the errors?\"

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

lang  words  mean_subwords  pct_words_multi_subword  pct_words_with_unk
  hi  19893          1.124                    8.194               0.015
  mr  11833          1.315                   20.350               0.008
  bn   8791          1.305                   18.087               0.046
lang   factor          level  entities  hindi_model_recall  in_language_recall
  hi   script          latin        13              0.7436                 NaN
  hi   script mixed_or_other         2              1.0000                 NaN
  hi   script         native      1914              0.8561                 NaN
  hi subwords              1       593              0.8735                 NaN
  hi subwords              2       568              0.8533                 NaN
  hi subwords            3-4       592              0.8502                 NaN
  hi subwords             5+       176              0.8201                 NaN
  hi  has_unk         no_unk      1929              0.8555                 NaN
 

## Save outputs

Predictions and error rows are compressed, and the summary tables are plain CSV. `results.csv` must still be unchanged.

In [14]:
def write_gzip_text(path, writer):  # helper: opens a gzip file as a UTF-8 text stream and lets `writer` fill it, for deterministic, reproducible compressed output
## Why filename="" and mtime=0: gzip normally embeds the original filename and the current timestamp in its header, so two runs producing byte-for-byte identical content would still produce DIFFERENT compressed files; pinning both to constants makes the compressed output reproducible, so a file-hash-based "did this change" check would actually work on it.
    with open(path, "wb") as raw, gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as gz, \
            io.TextIOWrapper(gz, encoding="utf-8", newline="\n") as text:  # filename="" and mtime=0 make the gzip bytes reproducible (no embedded filename or timestamp)
        writer(text)  # delegate the actual content-writing to the caller-supplied function


def write_predictions(text):  # writes every system's predictions for every sentence as JSON Lines, for later inspection without re-running any model
    for (sid, lang), pred in sorted(PREDS.items()):  # sorted for a deterministic file order across runs
        for i, tags in enumerate(pred):  # write one JSON record per sentence for this system
            text.write(json.dumps(dict(system=sid, lang=lang, idx=i, pred="".join(str(label2id[t]) for t in tags)),  # build this sentence's record (continued below): tags packed as a compact string of label ids
                                  ensure_ascii=False, separators=(",", ":")) + "\n")  # tags stored as a compact digit string (one label id per position) to keep the file small


write_gzip_text(os.path.join(OUT_DIR, "predictions.jsonl.gz"), write_predictions)  # write every system's full predictions, compressed
write_gzip_text(os.path.join(OUT_DIR, "errors.csv.gz"), lambda text: ERR.to_csv(text, index=False))  # write the full per-entity error table, compressed (this can be a large file)
for name, frame, index in (("scores", SCORES, False), ("error_counts", COUNTS.reset_index(), False),  # write each summary table to its own CSV file (continued below)
                           ("error_counts_by_type", TYPE_COUNTS.reset_index(), False),  # (continued) remaining tables to write out
                           ("transfer_gaps", TRANSFER_GAPS, False), ("consistent_wins", CONSISTENT_WINS, False),  # (continued) remaining tables to write out
                           ("token_stats", TOKEN_STATS, False), ("script_recall", SCRIPT_RECALL, False)):  # the smaller summary tables, written as plain (uncompressed) CSV for easy downstream reading
    frame.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=index, encoding="utf-8", lineterminator="\n")  # reset_index() first for the groupby-derived tables so their index columns are written out too

if file_sha(RESULTS_CSV) != RESULTS_SHA:  # final check of the read-only guard taken near the top of the notebook
    raise RuntimeError("results/results.csv changed during this run")  # catches any accidental write to the shared scoreboard during this analysis run

run_info = dict(git=GIT_COMMIT, debug=DEBUG, results_csv_sha256=RESULTS_SHA, n_systems=len(META), n_scores=len(SCORES),  # assemble the provenance record (continued below): commit, data sizes, and checksums
                sentences={lang: len(rows) for lang, rows in gold.items()},  # (continued) how many sentences were scored per language
                bilstm_drive_id=BILSTM_DRIVE_ID, bilstm_sha256=BILSTM_SHA256,  # (continued) which BiLSTM-CRF checkpoint was used, by Drive id and verified hash
                versions={p: version(p) for p in ("transformers", "accelerate", "seqeval", "sentencepiece", "pytorch-crf")},  # (continued) installed library versions, for reproducibility
                torch=torch.__version__, label_map=LABELS)  # a complete provenance record: commit, library versions, data sizes, and checksums for this run
with open(os.path.join(OUT_DIR, "run_info.json"), "w", encoding="utf-8", newline="\n") as f:  # write the provenance record out as JSON
    json.dump(run_info, f, ensure_ascii=False, indent=1)  # saved alongside the CSVs so results can later be traced back to the exact code/data/library versions that produced them
print("saved:", sorted(os.listdir(OUT_DIR)))  # confirms every expected output file was written
print("results.csv unchanged")  # confirms the read-only guard passed
print("cross-stage error analysis complete | debug:", DEBUG)  # final completion marker for the notebook run

saved: ['consistent_wins.csv', 'error_counts.csv', 'error_counts_by_type.csv', 'errors.csv.gz', 'predictions.jsonl.gz', 'run_info.json', 'scores.csv', 'script_recall.csv', 'token_stats.csv', 'transfer_gaps.csv']
results.csv unchanged
cross-stage error analysis complete | debug: False


## Findings

Written after the real run, from the printed tables above. The write-up belongs in `docs/phase5/error_analysis.md` and
Sections 7 and 8 of `docs/phase6/report.md`.